# Smooth Cavity Icebase
In this script, the initial ice base (create in GenerateCavityFiles.ipynb) is smoothed using k-ring neighbors.

In [1]:
import so_ase as so
import numpy as np
from tqdm import tqdm
from collections import defaultdict
import cmocean.cm as cmo
import cartopy.crs as ccrs
import matplotlib.pyplot as plt

/albedo/home/fheukamp/miniforge3/envs/so_ase/lib/python3.13/site-packages/pyfesom2/ut.py:15: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/albedo/home/fheukamp/miniforge3/envs/so_ase/lib/python3.13/site-packages/pyfesom2/climatology.py:14: UserWarning: The seawater library is deprecated! Please use gsw instead.
  import seawater as sw


osgeo is not installed, conversion to Geo formats like Geotiff (fesom2GeoFormat) will not work.


In [2]:
plot=False

### Set paths to mesh files

In [3]:
meshpath_raw         = "/albedo/home/fheukamp/mesh/fmesh/mesh_jigsaw_test/mesh_RTopo2.0.4_30sec_SOCAV_v5/01_raw/"
meshpath_seafloor_raw  = "/albedo/home/fheukamp/mesh/fmesh/mesh_jigsaw_test/mesh_RTopo2.0.4_30sec_SOCAV_v5/02_smoothed_bathymetry/"
meshpath_cavity_raw  = "/albedo/home/fheukamp/mesh/fmesh/mesh_jigsaw_test/mesh_RTopo2.0.4_30sec_SOCAV_v5/03_cavity_depth/"

### Read mesh files

In [4]:
elements = so.read_elements(meshpath_raw)
node_lon, node_lat, node_idx, node_coast = so.read_nodes(meshpath_raw)

In [5]:
with open(f'{meshpath_raw}nod2d.out', 'r') as f:
    num_nodes = int(f.readline())
    with open(f'{meshpath_cavity_raw}cavity_depth.out', 'r') as f:
        depths = []
        for _ in range(num_nodes):
            parts = f.readline()
            d = float(parts)
            depths.append(d)

cavity_depths = np.array(depths)

In [6]:
with open(f'{meshpath_raw}nod2d.out', 'r') as f:
    num_nodes = int(f.readline())
    with open(f'{meshpath_seafloor_raw}depth_smooth.out', 'r') as f:
        depths = []
        for _ in range(num_nodes):
            parts = f.readline()
            d = float(parts)
            depths.append(d)

floor_depths = np.array(depths)

### Build nodal k-ring neighbors for smoothing

In [7]:
from collections import deque

def build_node_k_ring_neighbors(elements, node_idx, k):

    N = node_idx.size
    neighbors_1 = [set() for _ in range(N)]
    
    for a, b, c in elements:
        neighbors_1[a].update((b, c))
        neighbors_1[b].update((a, c))
        neighbors_1[c].update((a, b))
    
    N = len(neighbors_1)
    k_ring = [set() for _ in range(N)]

    for node in tqdm(range(N)):
        visited = {node}            # avoid including the center node
        queue = deque([(node, 0)])

        while queue:
            current, dist = queue.popleft()
            if dist == k:     # stop expanding
                continue

            for nb in neighbors_1[current]:
                if nb not in visited:
                    visited.add(nb)
                    k_ring[node].add(nb)   # add to result
                    queue.append((nb, dist + 1))
    
    return k_ring


In [28]:
ring_2 = build_node_k_ring_neighbors(elements, node_idx, k=2)

100%|██████████| 3222974/3222974 [00:31<00:00, 103419.62it/s]


In [9]:
ring_3 = build_node_k_ring_neighbors(elements, node_idx, k=3)

100%|██████████| 3222974/3222974 [01:05<00:00, 49394.14it/s]


In [18]:
ring_4 = build_node_k_ring_neighbors(elements, node_idx, k=4)

100%|██████████| 3222974/3222974 [01:56<00:00, 27672.20it/s]


### Smooth ice base

In [8]:
def laplacian_explicit_smoothing(f, neighbors, relax=1.0, mask='None', mask_value=0, conserve_mean=False):
    """
    Explicit Laplacian smoothing of scalar field f.
    neighbors: list of iterables; neighbors[i] are neighbor indices for node i.
    iterations: number of passes
    relax: relaxation factor (0..1). new = (1-relax)*f_old + relax*mean(neigh)
    conserve_mean: if True, re-normalize to original mean after each iteration
    """
    N = f.size
    orig_mean = f.mean()

    f_new = f.copy()

    if all(mask == 'None'):
        n = np.arange(N)
    else:
        n = np.arange(N)[mask]
        
    for i in tqdm(n):
    
        nb = list(neighbors[i])
        nb = [int(x) for x in nb]
        mask = f[nb] != mask_value
        nb = np.array(nb)[mask]
        if len(nb) == 0:
            f_new[i] = f[i]
        else:
            f_new[i] = (1.0-relax)*f[i] + relax*(f[nb].mean())
    f = f_new
    if conserve_mean:
        f += (orig_mean - f.mean())
    return f

In [34]:
cavity_depths_smlap = laplacian_explicit_smoothing(cavity_depths, ring_3, mask=(cavity_depths<0), mask_value=0)

100%|██████████| 79710/79710 [00:01<00:00, 60843.52it/s]


In [9]:
if plot:
    h = np.where(cavity_depths_smlap != 0)[0]
    
    fig, ax = plt.subplots(1,2, figsize=(30,15), subplot_kw=dict(projection=ccrs.SouthPolarStereo()))
    
    for axis in ax:
        so.create_map(axis, extent=[-180, 180, -90, -60])
    
    cb = ax[0].scatter(node_lon[h], node_lat[h], c=cavity_depths_smlap[h], s=1, transform=ccrs.PlateCarree())
    
    plt.colorbar(cb, shrink=.5, aspect=20, orientation='horizontal', pad=.05)
    
    cb = ax[1].scatter(node_lon[h], node_lat[h], c=cavity_depths[h] - cavity_depths_smlap[h], s=1, cmap='RdBu_r', vmin=-200, vmax=200, transform=ccrs.PlateCarree())
    
    plt.colorbar(cb, shrink=.5, aspect=20, orientation='horizontal', pad=.05)

### Smooth seafloor
Deepen the seafloor at locations where the seafloor is intersecting with the icebase by averaging all ring-3 neighbors which are deeper.

In [68]:
mask = (floor_depths >= cavity_depths_smlap)

floor_depths_ori = floor_depths.copy()
floor_depths_mod = floor_depths.copy()

itercount = 0
while mask.sum() > 0:
    itercount += 1
    print(f'Iteration #{itercount}')
    
    for i, d in enumerate(floor_depths_ori):
        if mask[i]:
            nb = list(ring_3[i])
            nb = [int(x) for x in nb]
            nb_depths = floor_depths_ori[nb]
            nb_depths_deeper = nb_depths[nb_depths < d]
            if np.isnan(np.mean(nb_depths_deeper)):
                for n in nb:
                    floor_depths_mod[n] = floor_depths_mod[i] - 10 # reduce depth of all neighbors
            else:                
                floor_depths_mod[i] = np.mean(nb_depths_deeper)
            

    floor_depths_ori = floor_depths_mod.copy()
    mask = (floor_depths_mod >= cavity_depths_smlap)
    

Iteration #1


/albedo/home/fheukamp/miniforge3/envs/so_ase/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/albedo/home/fheukamp/miniforge3/envs/so_ase/lib/python3.13/site-packages/numpy/_core/_methods.py:145: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


Iteration #2
Iteration #3
Iteration #4
Iteration #5
Iteration #6
Iteration #7
Iteration #8
Iteration #9
Iteration #10
Iteration #11
Iteration #12
Iteration #13
Iteration #14
Iteration #15
Iteration #16
Iteration #17


### Write final files

In [70]:
# cavity_depth_smoothed.out
destpath = '/albedo/home/fheukamp/mesh/fmesh/mesh_jigsaw_test/mesh_RTopo2.0.4_30sec_SOCAV_v5/03_cavity_depth/'
with open(f'{destpath}cavity_depth_smoothed.out', 'w') as f:
    for l in cavity_depths_smlap:
        f.write(f"{int(l)}.0\n")

In [71]:
# cavity_depth_smoothed.out
destpath = '/albedo/home/fheukamp/mesh/fmesh/mesh_jigsaw_test/mesh_RTopo2.0.4_30sec_SOCAV_v5/03_cavity_depth/'
with open(f'{destpath}depth_smoothed_ring_3.out', 'w') as f:
    for l in floor_depths_mod:
        f.write(f"{int(l)}.0\n")

In [10]:
if plot:
    h = (cavity_depths_smlap != 0)
    
    fig, ax = plt.subplots(1,3, figsize=(30,10), subplot_kw=dict(projection=ccrs.SouthPolarStereo()))
    
    for axis in ax:
        so.create_map(axis, extent=[-180, 180, -90, -60])
    
    ax[0].set_title("Floor Depth MOD")
    cb = ax[0].scatter(node_lon[h], node_lat[h], c=floor_depths_mod[h], s=1, cmap=cmo.deep_r, transform=ccrs.PlateCarree())
    plt.colorbar(cb, shrink=.5, aspect=20, orientation='horizontal', pad=.05)
    
    ax[1].set_title("Floor Depth RAW")
    cb = ax[1].scatter(node_lon[h], node_lat[h], c=floor_depths[h], s=1, cmap=cmo.deep_r, transform=ccrs.PlateCarree())
    plt.colorbar(cb, shrink=.5, aspect=20, orientation='horizontal', pad=.05)
    
    ax[2].set_title("Floor Depth DELTA")
    h = h & ((floor_depths_mod - floor_depths) != 0)
    cb = ax[2].scatter(node_lon[h], node_lat[h], c=floor_depths_mod[h] - floor_depths[h], s=1, cmap='Reds_r', vmin=-200, transform=ccrs.PlateCarree())
    plt.colorbar(cb, shrink=.5, aspect=20, orientation='horizontal', pad=.05)